# Week 2 — Advanced PyTorch Optimization and Introduction to CUDA

**Course:** High-Performance Computing & Scaling Large Models  
**Practical:** Writing a matrix multiplication (`Y = X · W`) CUDA kernel from scratch.

---

## Learning Objectives

1. Apply PyTorch-level optimizations: `torch.compile`, mixed precision, channels-last, fused optimizers.
2. Understand the **CUDA execution model**: grids, blocks, warps, and the memory hierarchy.
3. Write three increasingly sophisticated matrix-multiplication kernels:
   - A *naive* one-thread-per-output kernel.
   - A *tiled* kernel using shared memory.
   - A *vectorized* kernel using `float4` loads.
4. Benchmark our kernels against cuBLAS (`torch.matmul`) and quantify the gap.

## Prerequisites

- Week 1 material (roofline, profiling).
- A CUDA-capable GPU (Compute Capability ≥ 7.0 recommended).
- `nvcc` available in `PATH`. We will invoke it via PyTorch's `torch.utils.cpp_extension`.


## 1. PyTorch-Level Optimizations

Before reaching for CUDA, exhaust the optimizations PyTorch already offers. They are *free* in engineering cost and often deliver 2–4× speedups.

### 1.1 `torch.compile`

`torch.compile` (PyTorch 2.0+) traces a model with TorchDynamo and lowers it to fused kernels via TorchInductor. The pertinent observation: most transformer overhead lives in element-wise epilogues (`GELU`, `LayerNorm`, residual `+`). Inductor *fuses* these with their preceding GEMM, slashing memory traffic.

### 1.2 Mixed Precision (AMP)

Storing weights and activations in **bfloat16** (BF16) or **float16** (FP16) halves memory traffic and unlocks Tensor Core throughput. BF16 retains FP32's exponent range and is the de-facto default for training on Ampere and newer hardware.

### 1.3 Fused Optimizers

`torch.optim.AdamW(..., fused=True)` collapses the per-parameter update kernels into a single CUDA call. For a model with thousands of parameter tensors, this turns thousands of kernel launches into one.

We will measure the impact of each below.


In [ ]:
import os, time, math
import numpy as np
import torch
import torch.nn as nn

assert torch.cuda.is_available(), "This notebook requires a CUDA device."
device = torch.device("cuda")
torch.manual_seed(0)
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"Device   : {torch.cuda.get_device_name(0)}")


In [ ]:
# A small transformer for benchmarking
class TinyTransformer(nn.Module):
    def __init__(self, d=1024, n_heads=8, n_layers=4):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d, n_heads, 4 * d, batch_first=True, norm_first=True, dropout=0.0
        )
        self.body = nn.TransformerEncoder(layer, n_layers)
        self.head = nn.Linear(d, d)

    def forward(self, x):
        return self.head(self.body(x))


B, L, D = 8, 512, 1024
x       = torch.randn(B, L, D, device=device)
model   = TinyTransformer(D).to(device)


def bench(fn, n_iter=50, warmup=10):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(n_iter):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / n_iter   # ms


def fwd():
    with torch.no_grad():
        _ = model(x)

print(f"Eager FP32          : {bench(fwd):.3f} ms")


In [ ]:
# Mixed precision
def fwd_amp():
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        _ = model(x)
print(f"Eager BF16 (AMP)    : {bench(fwd_amp):.3f} ms")


In [ ]:
# torch.compile — first call traces and compiles, so we time after warmup
compiled = torch.compile(model, mode="reduce-overhead", fullgraph=False)
def fwd_compiled():
    with torch.no_grad():
        _ = compiled(x)

# Larger warmup to let the compile finish
print(f"Compiled FP32       : {bench(fwd_compiled, warmup=20):.3f} ms")

def fwd_compiled_amp():
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        _ = compiled(x)
print(f"Compiled BF16       : {bench(fwd_compiled_amp, warmup=20):.3f} ms")


You should see speedups roughly in the ratio **1 : 1.7 : 2.0 : 3.0** for the four configurations on Ampere-class hardware. Two takeaways:

- *Mixed precision alone* yields nearly 2× — and almost no code change.
- *Compilation* fuses element-wise epilogues into the GEMMs, eliminating most LayerNorm / activation memory traffic.

We now turn to CUDA itself.


## 2. The CUDA Execution Model

A CUDA program organizes parallelism as a three-level hierarchy:

```
Grid                                ←  the entire kernel launch
 └── Block      (≤ 1024 threads)    ←  scheduled to one SM, shares memory
      └── Warp  (32 threads)        ←  the unit of SIMT execution
           └── Thread               ←  one logical lane
```

Three rules govern performance:

1. **Coalesced access.** Threads in a warp issue memory transactions together. If their addresses are contiguous, the hardware merges 32 requests into one 128-byte transaction. Strided or random access *serializes* and crushes bandwidth.
2. **Shared memory tiling.** Each SM has ~192 KB of fast on-chip storage (`__shared__`). Loading a tile of inputs there *once* and reusing it across many threads is the canonical optimization for GEMM, convolutions, and reductions.
3. **Occupancy.** The fraction of an SM's maximum thread complement that we actually use. Limited by register pressure, shared memory consumption, and block size.

A *kernel* is a function executed by every thread in the grid. Each thread receives its identity through built-in variables:

```cpp
int row = blockIdx.y * blockDim.y + threadIdx.y;
int col = blockIdx.x * blockDim.x + threadIdx.x;
```

The host (CPU) launches the kernel with `<<<grid, block>>>` syntax.

We will now write three matmul kernels of progressively increasing sophistication.


## 3. Matrix Multiplication, From Scratch

We compute $Y = X \cdot W$ where $X \in \mathbb{R}^{M \times K}$ and $W \in \mathbb{R}^{K \times N}$. For a single output element:

$$
Y_{ij} = \sum_{k=0}^{K-1} X_{ik} \, W_{kj}
$$

This is the fundamental operation behind every `nn.Linear` in your model.


In [ ]:
# Kernel source. We use PyTorch's inline JIT compiler.
from torch.utils.cpp_extension import load_inline

cuda_source = r'''
#include <cuda_runtime.h>
#include <torch/extension.h>

// ---------------- Kernel 1: Naive ----------------
__global__ void matmul_naive_kernel(
    const float* __restrict__ X,
    const float* __restrict__ W,
    float* __restrict__ Y,
    int M, int N, int K) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row >= M || col >= N) return;
    float acc = 0.f;
    for (int k = 0; k < K; ++k) {
        acc += X[row * K + k] * W[k * N + col];
    }
    Y[row * N + col] = acc;
}

// ---------------- Kernel 2: Tiled with shared memory ----------------
#define TILE 32
__global__ void matmul_tiled_kernel(
    const float* __restrict__ X,
    const float* __restrict__ W,
    float* __restrict__ Y,
    int M, int N, int K) {
    __shared__ float Xs[TILE][TILE];
    __shared__ float Ws[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float acc = 0.f;

    for (int kb = 0; kb < (K + TILE - 1) / TILE; ++kb) {
        int x_col = kb * TILE + threadIdx.x;
        int w_row = kb * TILE + threadIdx.y;

        Xs[threadIdx.y][threadIdx.x] =
            (row < M && x_col < K) ? X[row * K + x_col] : 0.f;
        Ws[threadIdx.y][threadIdx.x] =
            (w_row < K && col < N) ? W[w_row * N + col] : 0.f;
        __syncthreads();

        #pragma unroll
        for (int k = 0; k < TILE; ++k) {
            acc += Xs[threadIdx.y][k] * Ws[k][threadIdx.x];
        }
        __syncthreads();
    }
    if (row < M && col < N) {
        Y[row * N + col] = acc;
    }
}

// ---------------- Host wrappers ----------------
torch::Tensor matmul_naive(torch::Tensor X, torch::Tensor W) {
    TORCH_CHECK(X.is_cuda() && W.is_cuda(), "Inputs must be CUDA tensors");
    TORCH_CHECK(X.dtype() == torch::kFloat32 && W.dtype() == torch::kFloat32,
                "FP32 only in this teaching kernel");
    int M = X.size(0), K = X.size(1), N = W.size(1);
    auto Y = torch::empty({M, N}, X.options());
    dim3 block(16, 16);
    dim3 grid((N + 15) / 16, (M + 15) / 16);
    matmul_naive_kernel<<<grid, block>>>(
        X.data_ptr<float>(), W.data_ptr<float>(), Y.data_ptr<float>(), M, N, K);
    return Y;
}

torch::Tensor matmul_tiled(torch::Tensor X, torch::Tensor W) {
    TORCH_CHECK(X.is_cuda() && W.is_cuda(), "Inputs must be CUDA tensors");
    TORCH_CHECK(X.dtype() == torch::kFloat32 && W.dtype() == torch::kFloat32,
                "FP32 only in this teaching kernel");
    int M = X.size(0), K = X.size(1), N = W.size(1);
    auto Y = torch::empty({M, N}, X.options());
    dim3 block(TILE, TILE);
    dim3 grid((N + TILE - 1) / TILE, (M + TILE - 1) / TILE);
    matmul_tiled_kernel<<<grid, block>>>(
        X.data_ptr<float>(), W.data_ptr<float>(), Y.data_ptr<float>(), M, N, K);
    return Y;
}
'''

cpp_source = r'''
torch::Tensor matmul_naive(torch::Tensor X, torch::Tensor W);
torch::Tensor matmul_tiled(torch::Tensor X, torch::Tensor W);
'''

print("Compiling CUDA extension (this may take 1–2 minutes the first time)...")
ext = load_inline(
    name="week2_matmul",
    cpp_sources=[cpp_source],
    cuda_sources=[cuda_source],
    functions=["matmul_naive", "matmul_tiled"],
    verbose=False,
)
print("Done.")


In [ ]:
# Correctness check
def check_close(a, b, atol=1e-3, rtol=1e-3):
    return torch.allclose(a, b, atol=atol, rtol=rtol)

M, K, N = 512, 1024, 768
X = torch.randn(M, K, device=device)
W = torch.randn(K, N, device=device)

Y_ref   = X @ W
Y_naive = ext.matmul_naive(X, W)
Y_tiled = ext.matmul_tiled(X, W)

print(f"Naive matches reference : {check_close(Y_naive, Y_ref)}")
print(f"Tiled matches reference : {check_close(Y_tiled, Y_ref)}")
print(f"Max |Y_naive - Y_ref|   : {(Y_naive - Y_ref).abs().max().item():.2e}")
print(f"Max |Y_tiled - Y_ref|   : {(Y_tiled - Y_ref).abs().max().item():.2e}")


## 4. Benchmarking Against cuBLAS

`torch.matmul` dispatches to cuBLAS — a hand-tuned library that uses Tensor Cores, hierarchical tiling, and architecture-specific code paths. It is the ceiling we measure ourselves against.


In [ ]:
# Benchmark all three across square problem sizes
def bench_kernel(fn, X, W, n_iter=20, warmup=5):
    for _ in range(warmup):
        _ = fn(X, W)
    torch.cuda.synchronize()
    s = torch.cuda.Event(enable_timing=True); e = torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(n_iter):
        _ = fn(X, W)
    e.record()
    torch.cuda.synchronize()
    ms = s.elapsed_time(e) / n_iter
    M, K = X.shape; _, N = W.shape
    tflops = (2 * M * N * K) / (ms / 1000) / 1e12
    return ms, tflops


print(f"{'N':>6} | {'naive (ms)':>12} {'TF/s':>6}  |  {'tiled (ms)':>12} {'TF/s':>6}  |  {'cuBLAS (ms)':>13} {'TF/s':>6}")
print("-" * 92)
for N in (256, 512, 1024, 2048):
    X = torch.randn(N, N, device=device)
    W = torch.randn(N, N, device=device)
    n_ms, n_tf = bench_kernel(ext.matmul_naive, X, W)
    t_ms, t_tf = bench_kernel(ext.matmul_tiled, X, W)
    b_ms, b_tf = bench_kernel(torch.matmul,      X, W)
    print(f"{N:>6} | {n_ms:>12.3f} {n_tf:>6.2f}  |  "
          f"{t_ms:>12.3f} {t_tf:>6.2f}  |  {b_ms:>13.3f} {b_tf:>6.2f}")


**Reading the table.**

- The **naive** kernel typically attains 0.1–0.5 TFLOPs — a small fraction of FP32 peak (19.5 TFLOPs on A100). Every output performs $2K$ FLOPs against $2K$ FP32 loads (8K bytes), giving an arithmetic intensity of $\tfrac{1}{4}$ FLOP/byte. Catastrophic.
- The **tiled** kernel reuses each loaded tile $\text{TILE}$ times, multiplying arithmetic intensity by the tile size. We typically observe a 5–10× improvement.
- **cuBLAS** still wins by another 3–5×. It uses Tensor Cores (TF32 on Ampere), hierarchical tiling at the warp level, double-buffered loads, and architecture-tuned tile sizes.

The lesson is not that our kernel is bad — it is that *closing the last 80 % of the gap requires hardware-specific engineering* that cuBLAS, CUTLASS, and Triton invest in continuously. For most projects, the right move is to **use** these libraries; for the small set of operations they don't cover (custom attention masks, exotic data layouts, sparsity patterns), we write our own.


## 5. Going Further: A Glimpse of Triton

Writing CUDA by hand is illuminating but slow. **Triton** (Tillet, 2019) is a Python DSL that compiles to efficient GPU kernels. The same tiled matmul in Triton fits in roughly 25 lines, achieves cuBLAS-level performance, and is portable across hardware generations.

We will not require Triton in this course, but every modern kernel developer should be aware of it. FlashAttention (Week 3) ships both CUDA and Triton implementations.


In [ ]:
# (Optional) Triton matmul, for students with Triton installed.
try:
    import triton
    import triton.language as tl

    @triton.jit
    def matmul_triton_kernel(X, W, Y, M, N, K,
                             stride_xm, stride_xk,
                             stride_wk, stride_wn,
                             stride_ym, stride_yn,
                             BLOCK_M: tl.constexpr,
                             BLOCK_N: tl.constexpr,
                             BLOCK_K: tl.constexpr):
        pid_m = tl.program_id(0)
        pid_n = tl.program_id(1)
        offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        offs_k = tl.arange(0, BLOCK_K)

        x_ptrs = X + offs_m[:, None] * stride_xm + offs_k[None, :] * stride_xk
        w_ptrs = W + offs_k[:, None] * stride_wk + offs_n[None, :] * stride_wn

        acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
        for k in range(0, K, BLOCK_K):
            x = tl.load(x_ptrs, mask=(offs_m[:, None] < M) & (k + offs_k[None, :] < K), other=0.)
            w = tl.load(w_ptrs, mask=(k + offs_k[:, None] < K) & (offs_n[None, :] < N), other=0.)
            acc += tl.dot(x, w)
            x_ptrs += BLOCK_K * stride_xk
            w_ptrs += BLOCK_K * stride_wk

        y_ptrs = Y + offs_m[:, None] * stride_ym + offs_n[None, :] * stride_yn
        tl.store(y_ptrs, acc, mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))

    def matmul_triton(X, W):
        M, K = X.shape
        _, N = W.shape
        Y = torch.empty((M, N), device=X.device, dtype=X.dtype)
        grid = (triton.cdiv(M, 64), triton.cdiv(N, 64))
        matmul_triton_kernel[grid](
            X, W, Y, M, N, K,
            X.stride(0), X.stride(1),
            W.stride(0), W.stride(1),
            Y.stride(0), Y.stride(1),
            BLOCK_M=64, BLOCK_N=64, BLOCK_K=32,
        )
        return Y

    X = torch.randn(1024, 1024, device=device)
    W = torch.randn(1024, 1024, device=device)
    Y_tr = matmul_triton(X, W)
    print(f"Triton kernel matches cuBLAS: {torch.allclose(Y_tr, X @ W, atol=1e-2)}")
    ms, tf = bench_kernel(matmul_triton, X, W)
    print(f"Triton @ N=1024 : {ms:.3f} ms ({tf:.2f} TFLOPs/s)")
except ImportError:
    print("Triton not installed; skipping. Install with: pip install triton")


## 6. Exercises

**Exercise 2.1 — Occupancy.**  
Use `cudaOccupancyMaxActiveBlocksPerMultiprocessor` (in a small CUDA test program) to compute the theoretical occupancy of `matmul_tiled_kernel` for `TILE=16`, `TILE=32`, `TILE=64`. Which gives the highest occupancy, and does that correspond to the best runtime?

**Exercise 2.2 — Vectorized loads.**  
Modify the tiled kernel to load `float4` (16 bytes) per thread instead of `float` (4 bytes). What is the expected speedup, and what is the observed one?

**Exercise 2.3 — Mixed precision.**  
Re-implement `matmul_tiled` to accept FP16 inputs with FP32 accumulation. Compare to `torch.matmul(X.half(), W.half())`.

**Exercise 2.4 — Profiling.**  
Run `ncu --set full python scripts/bench_matmul.py` and report the *Speed of Light* values for your tiled kernel. Where is it bound?


## 7. Summary

PyTorch offers free, layered optimizations — mixed precision, fused optimizers, `torch.compile` — that should always be applied first. When we need to go further, CUDA gives us direct control over the memory hierarchy. We saw that a naive matmul leaves more than 95 % of the device's compute on the floor, and that *shared-memory tiling* reclaims most of it. Closing the final gap to cuBLAS requires Tensor Core programming and architecture-specific tuning that production libraries handle for us.

Week 3 applies these ideas to a problem where libraries until recently *failed* us: the attention mechanism, whose memory profile became the central bottleneck of LLM training and serving.
